# Notebook 15 — Final Reproducibility Check

## Goal
Audit all artifacts before writing the paper.
Every required file must exist, every approval must be confirmed, every assertion must pass.

**If anything fails here, find the source notebook and rerun it.**

---

## What you must inspect
- All approved artifacts exist and are non-empty
- All approval JSON files are present and have `approved: true`
- All model files exist
- All paper tables and figures exist
- The full reproducibility checklist is complete

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')

def check_file(path, label, min_size_bytes=100):
    p = pathlib.Path(path)
    exists = p.exists()
    size = p.stat().st_size if exists else 0
    ok = exists and size >= min_size_bytes
    status = 'OK' if ok else ('EMPTY' if exists else 'MISSING')
    return {'label': label, 'path': str(p.relative_to(DRIVE_ROOT)), 'status': status, 'size_kb': size // 1024}

def check_approval(path, notebook_label):
    p = pathlib.Path(path)
    if not p.exists():
        return {'label': notebook_label, 'status': 'MISSING', 'approved': False}
    with open(p) as f:
        ap = json.load(f)
    approved = ap.get('approved', False)
    return {'label': notebook_label, 'status': 'OK' if approved else 'NOT_APPROVED', 'approved': approved}

print('Reproducibility check initialized.')
print(f'Drive root: {DRIVE_ROOT}')

## Check 1 — All approval files

In [ ]:
APPROVALS_DIR = DRIVE_ROOT / 'approvals'

approval_checks = [
    (APPROVALS_DIR / '01_raw_fred_alfred_approved.json', 'NB01: FRED/ALFRED'),
    (APPROVALS_DIR / '02_raw_gdelt_approved.json', 'NB02: GDELT'),
    (APPROVALS_DIR / '03_macro_vintage_approved.json', 'NB03: Macro vintage'),
    (APPROVALS_DIR / '04_news_cleaning_approved.json', 'NB04: News cleaning'),
    (APPROVALS_DIR / '05_news_features_approved.json', 'NB05: News features'),
    (APPROVALS_DIR / '06_macro_features_target_approved.json', 'NB06: Macro features'),
    (APPROVALS_DIR / '07_model_ready_dataset_approved.json', 'NB07: Model-ready dataset'),
    (APPROVALS_DIR / '08_baseline_models_approved.json', 'NB08: Baselines'),
    (APPROVALS_DIR / '09_xgboost_models_approved.json', 'NB09: XGBoost models'),
    (APPROVALS_DIR / '11_rag_retrieval_approved.json', 'NB11: RAG retrieval'),
    (APPROVALS_DIR / '12_deepseek_explanations_approved.json', 'NB12: Explanations'),
    (APPROVALS_DIR / '13_rag_evaluation_approved.json', 'NB13: RAG evaluation'),
]

approval_results = [check_approval(p, label) for p, label in approval_checks]
approval_df = pd.DataFrame(approval_results)

print('=== Approval Status ===')
print(approval_df.to_string(index=False))

all_approved = (approval_df['approved'] == True).all()
missing_approvals = approval_df[~approval_df['approved']]
if len(missing_approvals) > 0:
    print(f'\nWARNING: {len(missing_approvals)} notebooks not yet approved:')
    print(missing_approvals['label'].tolist())
else:
    print('\nAll notebooks approved.')

## Check 2 — All data artifacts

In [ ]:
artifact_checks = [
    (DRIVE_ROOT / 'data' / 'raw' / 'fred' / 'fred_current.parquet', 'FRED current data'),
    (DRIVE_ROOT / 'data' / 'raw' / 'alfred' / 'alfred_vintage.parquet', 'ALFRED vintage data'),
    (DRIVE_ROOT / 'data' / 'raw' / 'gdelt' / 'news_raw.parquet', 'GDELT raw news'),
    (DRIVE_ROOT / 'data' / 'interim' / 'macro' / 'macro_vintage_audited.parquet', 'Macro audited'),
    (DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_approved.parquet', 'Clean news'),
    (DRIVE_ROOT / 'data' / 'features' / 'news' / 'monthly_news_features_approved.parquet', 'News features'),
    (DRIVE_ROOT / 'data' / 'features' / 'macro' / 'macro_features_target_approved.parquet', 'Macro features'),
    (DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet', 'Model-ready dataset'),
]

artifact_results = [check_file(p, label) for p, label in artifact_checks]
artifact_df = pd.DataFrame(artifact_results)

print('=== Data Artifact Status ===')
print(artifact_df.to_string(index=False))

missing_artifacts = artifact_df[artifact_df['status'] != 'OK']
if len(missing_artifacts) > 0:
    print(f'\nWARNING: {len(missing_artifacts)} artifacts missing or empty:')
    print(missing_artifacts['label'].tolist())
else:
    print('\nAll data artifacts present.')

## Check 3 — Model files and predictions

In [ ]:
model_checks = [
    (DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_macro_only.joblib', 'XGBoost macro-only'),
    (DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_news_only.joblib', 'XGBoost news-only'),
    (DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_macro_news.joblib', 'XGBoost macro+news'),
    (DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_naive.parquet', 'Naive predictions'),
    (DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_arima.parquet', 'ARIMA predictions'),
    (DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_only.parquet', 'Macro-only predictions'),
    (DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_news_only.parquet', 'News-only predictions'),
    (DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_news.parquet', 'Macro+news predictions'),
    (DRIVE_ROOT / 'outputs' / 'metrics' / 'baseline_metrics.csv', 'Baseline metrics'),
    (DRIVE_ROOT / 'outputs' / 'metrics' / 'model_comparison.csv', 'Model comparison'),
    (DRIVE_ROOT / 'outputs' / 'metrics' / 'model_metrics.json', 'Model metrics JSON'),
]

model_results = [check_file(p, label) for p, label in model_checks]
model_df_check = pd.DataFrame(model_results)

print('=== Model and Prediction Files ===')
print(model_df_check.to_string(index=False))

missing_models = model_df_check[model_df_check['status'] != 'OK']
if len(missing_models) > 0:
    print(f'\nWARNING: {len(missing_models)} model files missing:')
    print(missing_models['label'].tolist())
else:
    print('\nAll model and prediction files present.')

## Check 4 — RAG artifacts

In [ ]:
rag_checks = [
    (DRIVE_ROOT / 'outputs' / 'explanations' / 'retrieved_evidence.parquet', 'Retrieved evidence'),
    (DRIVE_ROOT / 'outputs' / 'explanations' / 'forecast_explanations.parquet', 'Forecast explanations'),
    (DRIVE_ROOT / 'outputs' / 'tables' / 'rag_evaluation_template.csv', 'RAG evaluation template'),
]

rag_results = [check_file(p, label) for p, label in rag_checks]
rag_df = pd.DataFrame(rag_results)

print('=== RAG Artifacts ===')
print(rag_df.to_string(index=False))

# Load and verify evidence time filter
try:
    ev = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'explanations' / 'retrieved_evidence.parquet')
    ev['forecast_date'] = pd.to_datetime(ev['forecast_date'])
    ev['article_date'] = pd.to_datetime(ev['article_date'])
    leakage = ev[ev['article_date'] > ev['forecast_date']]
    if len(leakage) > 0:
        print(f'\nCRITICAL WARNING: {len(leakage)} evidence rows have article_date > forecast_date (RAG LEAKAGE)!')
    else:
        print('\nRAG time-filter re-verified: all article_date <= forecast_date. OK.')
except Exception as e:
    print(f'Could not verify RAG evidence: {e}')

## Check 5 — Paper tables and figures

In [ ]:
paper_checks = [
    (DRIVE_ROOT / 'outputs' / 'tables' / 'table1_data_summary.csv', 'Table 1: Data summary'),
    (DRIVE_ROOT / 'outputs' / 'tables' / 'table3_model_comparison.csv', 'Table 3: Model comparison'),
    (DRIVE_ROOT / 'outputs' / 'tables' / 'worst_forecast_errors.csv', 'Table 4: Worst errors'),
    (DRIVE_ROOT / 'outputs' / 'tables' / 'table5_rag_example.csv', 'Table 5: RAG example'),
    (DRIVE_ROOT / 'outputs' / 'figures' / 'fig1_actual_vs_predicted_paper.png', 'Figure 1: Actual vs. predicted'),
    (DRIVE_ROOT / 'outputs' / 'figures' / 'fig2_news_trends_paper.png', 'Figure 2: News trends'),
    (DRIVE_ROOT / 'outputs' / 'figures' / 'fig3_feature_importance_paper.png', 'Figure 3: Feature importance'),
]

paper_results = [check_file(p, label) for p, label in paper_checks]
paper_df = pd.DataFrame(paper_results)

print('=== Paper Output Files ===')
print(paper_df.to_string(index=False))

missing_paper = paper_df[paper_df['status'] != 'OK']
if len(missing_paper) > 0:
    print(f'\nWARNING: {len(missing_paper)} paper output files missing. Run NB14.')
    print(missing_paper['label'].tolist())
else:
    print('\nAll paper output files present.')

## Final anti-leakage assertion (re-run on model-ready dataset)

In [ ]:
print('Running final leakage assertion on model-ready dataset...')

try:
    model_df = pd.read_parquet(DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet')
    model_df['forecast_date'] = pd.to_datetime(model_df['forecast_date'])
    model_df['target_month'] = pd.to_datetime(model_df['target_month'])

    leakage = model_df[model_df['target_month'] <= model_df['forecast_date']]
    assert len(leakage) == 0, (
        f'DATA LEAKAGE DETECTED: {len(leakage)} rows where target_month <= forecast_date'
    )
    print(f'  PASSED: target_month > forecast_date for all {len(model_df)} rows.')
except FileNotFoundError:
    print('  SKIPPED: model-ready dataset not found.')

## Final Checklist

Check each item manually. Change `True` for completed items.

```
[ ] FRED/ALFRED raw data collected and approved
[ ] GDELT raw data collected and approved
[ ] Macro vintage audited and approved
[ ] News cleaned and approved
[ ] News features approved
[ ] Macro features and target approved
[ ] Model-ready dataset approved
[ ] Baseline models trained and approved
[ ] XGBoost models trained and approved
[ ] Model audit completed (NB10)
[ ] RAG retrieval approved
[ ] Explanations generated and approved
[ ] RAG evaluation template created and manually scored
[ ] Paper tables and figures created (NB14)
[ ] Final leakage assertion passed
[ ] All artifact files non-empty
```

In [ ]:
checklist = {
    'FRED/ALFRED raw data collected': False,
    'GDELT raw data collected': False,
    'Macro vintage audited': False,
    'News cleaned and approved': False,
    'News features approved': False,
    'Macro target approved': False,
    'Model-ready dataset approved': False,
    'Baselines trained': False,
    'XGBoost models trained': False,
    'Model audit completed': False,
    'RAG retrieval completed': False,
    'DeepSeek/template explanations completed': False,
    'RAG evaluation template created': False,
    'Paper outputs created': False,
}

# Auto-populate from approval files
ap_map = {
    'FRED/ALFRED raw data collected': '01_raw_fred_alfred_approved.json',
    'GDELT raw data collected': '02_raw_gdelt_approved.json',
    'Macro vintage audited': '03_macro_vintage_approved.json',
    'News cleaned and approved': '04_news_cleaning_approved.json',
    'News features approved': '05_news_features_approved.json',
    'Macro target approved': '06_macro_features_target_approved.json',
    'Model-ready dataset approved': '07_model_ready_dataset_approved.json',
    'Baselines trained': '08_baseline_models_approved.json',
    'XGBoost models trained': '09_xgboost_models_approved.json',
    'RAG retrieval completed': '11_rag_retrieval_approved.json',
    'DeepSeek/template explanations completed': '12_deepseek_explanations_approved.json',
    'RAG evaluation template created': '13_rag_evaluation_approved.json',
}

for label, fname in ap_map.items():
    p = DRIVE_ROOT / 'approvals' / fname
    if p.exists():
        with open(p) as f:
            ap = json.load(f)
        checklist[label] = ap.get('approved', False)

# Check paper outputs
checklist['Paper outputs created'] = (DRIVE_ROOT / 'outputs' / 'tables' / 'table3_model_comparison.csv').exists()
checklist['Model audit completed'] = (DRIVE_ROOT / 'outputs' / 'figures' / 'actual_vs_predicted.png').exists()

print('=== Final Reproducibility Checklist ===')
all_done = True
for item, done in checklist.items():
    status = '✓' if done else '✗'
    print(f'  [{status}] {item}')
    if not done:
        all_done = False

print()
if all_done:
    print('ALL CHECKS COMPLETE. Ready to write the paper.')
else:
    incomplete = [k for k, v in checklist.items() if not v]
    print(f'{len(incomplete)} items not yet complete: {incomplete}')

## Save artifact registry

In [ ]:
all_checks = approval_results + artifact_results + model_results + rag_results + paper_results
registry_df = pd.DataFrame(all_checks)
registry_path = DRIVE_ROOT / 'outputs' / 'logs' / 'artifact_registry.csv'
registry_df.to_csv(registry_path, index=False)
print(f'Artifact registry saved: {registry_path}')
print(f'Total artifacts checked: {len(registry_df)}')
print(f'OK: {(registry_df["status"] == "OK").sum()} | Issues: {(registry_df["status"] != "OK").sum()}')
print()
print(f'Reproducibility check completed at {datetime.datetime.utcnow().isoformat()} UTC')
print('Notebook 15 complete.')